# Anomaly-Based Network Intrusion Detection System
## Notebook 01: Data Exploration and Analysis

**Author:** Deepanshu Garhkoti  
**Date:** September 15, 2026  
**Dataset:** UNSW-NB15  

### Objectives
1. Load and inspect the UNSW-NB15 dataset
2. Analyse data quality and missing values
3. Examine feature distributions and correlations
4. Investigate class imbalance and attack-type distribution
5. Generate visualisations for better understanding

## 1. Imports and Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join('..'))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

from src.data.loader import DataLoader
from src.utils.visualization import plot_class_distribution, plot_correlation_matrix

print('Libraries loaded successfully')

## 2. Load Data

In [ ]:
loader = DataLoader(dataset='unsw_nb15')
df = loader.load_raw_data()

print(f'Shape : {df.shape}')
print(f'Samples : {df.shape[0]:,}')
print(f'Features: {df.shape[1]}')
df.head()

## 3. Data-Quality Check

In [ ]:
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100
quality = pd.DataFrame({'Missing': missing, 'Missing %': missing_pct})
quality = quality[quality['Missing'] > 0].sort_values('Missing', ascending=False)

if quality.empty:
    print('No missing values found')
else:
    print(f'{len(quality)} columns with missing values')
    display(quality)

dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes} ({dupes/len(df)*100:.2f}%)')

## 4. Target Variable Analysis

In [ ]:
TARGET = 'label'   # binary: 0=normal, 1=attack
CAT_TARGET = 'attack_cat'  # multi-class attack category

print('Binary label distribution:')
label_counts = df[TARGET].value_counts()
label_pct = df[TARGET].value_counts(normalize=True) * 100
display(pd.DataFrame({'Count': label_counts, 'Percent': label_pct}))

print(f'Imbalance ratio: {label_counts.max()/label_counts.min():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
label_counts.plot(kind='bar', ax=axes[0], color=['steelblue','tomato'])
axes[0].set_title('Binary Class Count')
axes[0].set_xlabel('Label (0=Normal, 1=Attack)')
axes[0].set_ylabel('Count')

if CAT_TARGET in df.columns:
    cat_counts = df[CAT_TARGET].value_counts()
    cat_counts.plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Attack Category Distribution')
    axes[1].set_xlabel('Attack Category')
    axes[1].set_ylabel('Count')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 5. Numeric Feature Statistics

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Numeric: {len(num_cols)}  |  Categorical: {len(cat_cols)}')

display(df[num_cols[:15]].describe().T)

## 6. Correlation Matrix

In [ ]:
corr = df[num_cols].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            linewidths=0.3, cbar_kws={'shrink': 0.7})
plt.title('Feature Correlation Matrix (lower triangle)', fontsize=14)
plt.tight_layout()
plt.show()

high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.85:
            high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

print(f'Highly correlated pairs (|r|>0.85): {len(high_corr)}')
for a, b, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True)[:10]:
    print(f'  {a}  <->  {b}  : {r:.3f}')

## 7. Observations and Next Steps

In [ ]:
print('Key observations:')
print('  - Dataset shape and missing values reviewed above')
print('  - Binary class imbalance ratio documented')
print('  - Highly correlated features identified for potential removal')
print()
print('Next: 02_preprocessing.ipynb')